In [1]:
%matplotlib qt
import os
import shutil
import mne
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# import pickle
# import linecache

# def save_var(var, fname_pickle):
#     with open(fname_pickle, 'wb') as f:
#         pickle.dump(var, f, pickle.HIGHEST_PROTOCOL)
#     return None

# def load_var(fname_pickle):
#     with open(fname_pickle, 'rb') as f:
#         return pickle.load(f)

# def text2matrix(trial_name, subj_name, data_path, n_times=256):
#     '''
#     Read EEG data from trial files in UCI format and convert to an [n_ch, n_times] matrix.
#     '''
#     # Get filename
#     fname = os.path.join(data_path, subj_name, trial_name)
    
#     # Read EEG voltages in a table and reshape into a matrix
#     table = np.loadtxt(fname, dtype='str')
#     voltages = np.array([float(v) for v in table[..., 3]]) # voltages are given in the 4th column
#     eeg_mat = np.reshape(voltages, (n_times, -1)).T
    
#     # Get trial condition from file header
#     hdr = linecache.getline(fname, 4) # trial condition is given in the 4th line
#     cond = hdr[2:6]
#     if "err" not in hdr:
#         if "S1" in cond:
#             event_id = 1
#         elif "S2 m" in cond:
#             event_id = 2
#         elif "S2 n" in cond:
#             event_id = 3
#         else:
#             raise ValueError('Event condition not identified.')
#     else:
#         if "S2 m" in cond:
#             event_id = 4
#         elif "S2 n" in cond:
#             event_id = 5
#         else:
#             raise ValueError('Event condition not identified.')
    
#     return eeg_mat, event_id

In [2]:
# Given
data_path = '/Users/chholakp2/data/aud'
dir_entries = os.scandir(data_path)
subj_names = sorted([entry.name for entry in dir_entries if entry.is_dir()])

# 1: Find total number of trials for all subjects

In [3]:
meta_data = {}
for subj_name in subj_names:
    trials = os.listdir(os.path.join(data_path, subj_name))

    trial_inds = sorted([int(trial[-3:]) for trial in trials])

    meta_data[subj_name] = dict(trial_indices=np.array(trial_inds),
                            total_trials=len(trial_inds))

# meta_data

In [4]:
n_tot_trials = [meta_data[subj_name]['total_trials'] for subj_name in subj_names]
print(n_tot_trials)

[88, 93, 74, 118, 118, 106, 115, 85, 100, 112, 108, 87, 104, 114, 100, 70, 72, 67, 87, 104, 96, 111, 98, 77, 96, 78, 92, 97, 96, 72, 61, 93, 106, 110, 94, 90, 115, 79, 85, 99, 101, 117, 100, 74, 77, 102, 30, 70, 89, 79, 59, 108, 104, 81, 74, 79, 109, 111, 101, 102, 92, 83, 80, 100, 93, 78, 104, 81, 83, 99, 115, 93, 112, 112, 89, 73, 92, 61, 92, 67, 99, 88, 69, 83, 110, 98, 102, 107, 59, 60, 106, 78, 100, 102, 109, 105, 88, 98, 66, 99, 59, 41, 99, 109, 85, 104, 57, 79, 103, 66, 119, 117, 86, 68, 58, 80, 71, 101, 98, 116, 71, 111]


In [5]:
sum(n_tot_trials)

11057

In [6]:
plt.figure()
plt.hist(n_tot_trials, bins=24, edgecolor='black', alpha=0.5)
plt.xlabel('Total no. of trials')
plt.ylabel('Total no. of subjects')
plt.show()

# 2: Check if order of channels is same throughout for all channels

In [7]:
ch_names_list = []
trials_list = []
for subj_name in subj_names:
    trials = os.listdir(os.path.join(data_path, subj_name))
    for trial in trials:
        trials_list.append(trial)
        fname = os.path.join(data_path, subj_name, trial)
        ch_names = np.loadtxt(fname, dtype='str', usecols=1)
        ch_names_list.append(ch_names.tolist())

In [8]:
ch_names_arr = np.array(ch_names_list)
print(np.shape(ch_names_arr))

(11057, 16384)


In [9]:
(ch_names_arr == ch_names_arr[0]).all()

True